# W01｜HTTP / FastAPI / Pydantic / Structured Output｜教学版

**建议学习时间：4h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 HTTP / Path&Query / Pydantic 基础 | 完成 Ch1/2/3/4/6 指定小节 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Agent Loop + Structured Output | 定义 Intent JSON schema | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 手写一个 /query 小接口 | 合法/非法请求都能正确响应 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 实现 question → structured intent | 至少 5 个问题可返回结构化结果 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 画 1 张请求链路图 | 不用 AI 讲清整条请求链路 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜`/query` 小接口参考答案
```python
class QueryResponse(BaseModel):
    intent: str
    metric: str | None = None
    dimension: str | None = None
    needs_clarification: bool

@app.post("/query", response_model=QueryResponse)
def query(req: QueryRequest):
    intent = mock_understand(req.question)
    return QueryResponse(
        **intent.model_dump(),
        needs_clarification=intent.intent == "unknown"
    )
```
验收：正常问题 `200`；`question="x"` 被 Pydantic 拒绝为 `422`。

### Task 4｜5 个 structured intent 验收样例
| Question | Expected intent | metric | dimension |
|---|---|---|---|
| Show monthly message trend | trend | message_count | month |
| Compare channels | comparison | message_count | channel |
| Top use cases | ranking | message_count | use_case |
| Show monthly trend please | trend | message_count | month |
| Hello | unknown | - | - |

### Task 5｜请求链路参考图
```text
Client
  ↓ POST /query
FastAPI Router
  ↓
Pydantic Request Validation
  ├─ invalid → 422
  └─ valid
      ↓
Intent / Query Logic
      ↓
Structured Intent
      ↓
Pydantic Response Model
      ↓
HTTP JSON Response
```

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 建立 Web Backend 最基本的 request/response 心智模型
- 会区分 Path / Query / Body
- 会用 Pydantic 定义输入输出契约
- 理解 Structured Output 为什么是 AI 工程的基础

        ## 本周前置检查

        - 会运行 Python/Jupyter
- 知道函数和 class 的基本语法；如果不会 type hint，本周一起学

        ## 本周必须记住的英文术语

        - `endpoint`：接口端点
- `path parameter`：路径参数
- `query parameter`：查询参数
- `request body`：请求体
- `validation`：校验
- `structured output`：结构化输出

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. HTTP 请求到底是什么？

    **是什么（What）**  
    浏览器、前端或其他服务通过 HTTP 向后端发送一个 Request，后端处理后返回 Response。

    **为什么需要（Why）**  
    FastAPI 只是帮助你更方便地接收和返回 HTTP；如果 HTTP 心智模型不清晰，后面 Path、Query、Body、Status Code 都会混在一起。

    **什么时候用（When）**  
    `GET` 查询资源；`POST` 创建/提交复杂输入；`PATCH` 部分更新；`DELETE` 删除。

    **常见误区 / 容易踩坑**  
    不要把 GET/POST 当成“只是不同装饰器”。它们表达不同语义；状态码也不是随便返回。


**中英术语**：`request` = 请求；`response` = 响应；`status code` = 状态码；`endpoint` = 接口端点

### 代码实验：Path / Query / Body 的区别

**这段代码想让你看什么？**


运行后重点看：
1. `/jobs/1` 中的 `1` 为什么属于 Path；
2. `/jobs?status=RUNNING` 中的 `status` 为什么属于 Query；
3. POST JSON 为什么属于 Body；
4. 非法类型为什么会在进入业务逻辑前就被 FastAPI/Pydantic 拒绝。


> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI()

FAKE_JOBS = {
    1: {"id": 1, "status": "RUNNING"},
    2: {"id": 2, "status": "DONE"},
}

class QueryRequest(BaseModel):
    question: str = Field(min_length=3, max_length=200)
    limit: int = Field(default=20, ge=1, le=100)

@app.get("/jobs/{job_id}")
def get_job(job_id: int):
    if job_id not in FAKE_JOBS:
        raise HTTPException(status_code=404, detail="Job not found")
    return FAKE_JOBS[job_id]

@app.get("/jobs")
def list_jobs(status: str | None = None, limit: int = Query(20, ge=1, le=100)):
    rows = list(FAKE_JOBS.values())
    if status:
        rows = [x for x in rows if x["status"] == status]
    return rows[:limit]

@app.post("/query")
def submit_query(req: QueryRequest):
    return {"received": req.model_dump()}

client = TestClient(app)

print("Path:", client.get("/jobs/1").status_code, client.get("/jobs/1").json())
print("Query:", client.get("/jobs?status=RUNNING").json())
print("Bad path:", client.get("/jobs/abc").status_code)
print("Bad body:", client.post("/query", json={"question": "x"}).status_code)

### Checkpoint A：先确认你看懂上面的代码

1. `job_id` 为什么更适合作为 Path Parameter，而不是 Query Parameter？
2. `status` 为什么更适合作为 Query Parameter？
3. `question='x'` 为什么还没进入 submit_query 就失败了？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** `job_id` 标识具体资源，URL `/jobs/1` 表示“编号 1 的 job”；这是资源身份的一部分。

**2.** `status` 是对一组 jobs 的筛选条件，不改变资源身份。

**3.** Pydantic 在 FastAPI 调用 endpoint 前已经根据 `min_length=3` 做了输入校验。

    </details>

## 2. Pydantic 为什么对 AI 应用特别重要？

    **是什么（What）**  
    Pydantic 把自由的 Python dict / JSON 转成有类型、有约束的结构化对象。

    **为什么需要（Why）**  
    AI 输出天然不稳定。生产程序不能靠“模型大概会返回这个 JSON”运行，必须用 schema 把 AI 输出限制成程序能验证的契约。

    **什么时候用（When）**  
    API Request/Response、LLM Structured Output、Tool 参数、配置对象。

    **常见误区 / 容易踩坑**  
    Pydantic 不是数据库；它是数据验证与 schema 层。也不要把 ORM model 直接当成所有 API response。


**中英术语**：`schema` = 结构定义；`validation` = 校验；`structured output` = 结构化输出

### 代码实验：让“自然语言理解结果”变成可验证的 Intent

**这段代码想让你看什么？**


重点观察：`Intent` 只允许预定义类型；这就是以后 LLM 输出与程序之间的契约。
现在我们先用 mock，不接真实 LLM，因为这周要学的是结构，不是模型调用。


> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Intent(BaseModel):
    intent: Literal["trend", "comparison", "ranking", "unknown"]
    metric: str | None = None
    dimension: str | None = None

def mock_understand(question: str) -> Intent:
    q = question.lower()
    if "compare" in q:
        return Intent(intent="comparison", metric="message_count", dimension="channel")
    if "top" in q:
        return Intent(intent="ranking", metric="message_count", dimension="use_case")
    if "trend" in q or "monthly" in q:
        return Intent(intent="trend", metric="message_count", dimension="month")
    return Intent(intent="unknown")

for q in ["Show monthly trend", "Compare channels", "Top use cases", "Hello"]:
    print(q, "->", mock_understand(q).model_dump())

### 练习：给 API 增加更完整的 Response Model


    新建 `QueryResponse`，字段至少包含：
    - `intent`
    - `metric`
    - `dimension`
    - `needs_clarification: bool`

    然后让 `/query` 返回这个 model。

<details>
<summary><b>提示</b></summary>

`needs_clarification` 可以根据 intent 是否为 `unknown` 决定。

</details>

In [ ]:
# TODO：先自己完成
# class QueryResponse(BaseModel):
#     ...

<details>
    <summary><b>参考答案</b></summary>

    ```python
class QueryResponse(BaseModel):
    intent: str
    metric: str | None = None
    dimension: str | None = None
    needs_clarification: bool

def build_response(question: str) -> QueryResponse:
    intent = mock_understand(question)
    return QueryResponse(
        **intent.model_dump(),
        needs_clarification=intent.intent == "unknown"
    )
```

    </details>

### 练习：判断错误属于 4xx 还是 5xx

分别判断：非法 limit、job 不存在、后端代码除零、数据库连接失败，应该更接近 4xx 还是 5xx？为什么？

<details>
    <summary><b>参考答案</b></summary>

    ```python
# 参考判断（不是代码题）：
# 非法 limit -> 4xx，因为客户端输入违反约束
# job 不存在 -> 404
# 后端除零 -> 500，服务端 bug
# DB 连接失败 -> 通常 5xx / 503，服务端依赖不可用
```

    </details>

## ChatBI 本周作业

        实现 `POST /api/query`：

`question → Pydantic validation → structured Intent → QueryResponse`

**本周禁止加数据库 / LangGraph / Redis。**  
目标只是把 API contract 和 structured output 学清楚。

### 本周面试 / 理解检查

1. 为什么 `/jobs/{job_id}` 用 Path，而 `?status=` 用 Query？
2. 为什么 AI 应用特别需要 Pydantic/Schema？
3. 422 与 500 的核心区别是什么？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** Path 标识具体资源；Query 通常用于筛选、排序、分页等查看方式。

**2.** 因为模型输出不稳定，程序必须在进入业务逻辑前验证结构、类型和允许值。

**3.** 422 更偏请求数据不满足 schema；500 表示服务器处理过程中发生未正确处理的内部错误。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。